# Step 10b – Figures of the ten classified maps and model agreement

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Section 3.5 · Figures 11–12, Table 9
- **Needs:** the classified map tiles in `DATA_DIR/maps/<model>/` from step 10a, and step 04
- **Writes:** figures in `DATA_DIR/maps/`, copies in `RESULTS_DIR/Figures_revised_14Oct/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / maps/
#   RESULTS_DIR / Figures_revised
#   RESULTS_DIR / Figures_revised.zip
#   RESULTS_DIR / Figures_revised_14Oct/
#   RESULTS_DIR / train_test_gap/

In [ ]:
!pip install -q catboost rasterio

In [ ]:
"""
Figures of the ten classified maps

Run after step 10a (the ten classified maps in DATA_DIR/maps/<model>/).

Figures (OUT_DIR, PNG 600 dpi + PDF):
  Fig_maps_all10          2 x 5 grid, one shared legend, inset area outlined
  Fig_maps_inset_all10    2 x 5 grid of the inset at full resolution
  Fig_maps_main           top three models + agreement map (2 x 2) for the main text
Table:
  model_agreement_summary.csv   share of mapped area where k of 10 models agree

Full-extent panels are resampled for display with the MODAL class (the 10 m maps are
too large to plot pixel by pixel). Agreement is computed on a systematic sample of the
original 10 m pixels (nearest-neighbour), so it is exact pixel-level agreement.

Revision for MDPI editor comment on Figure 10 (legend vs. figure content):
  - The inset square is drawn the SAME way in every panel (black outline with a thin
    white halo, so it stays visible on both the land-cover colours and the agreement
    map). Previously it was red in the agreement map but black in the legend.
  - "No data" is added to a legend ONLY if no-data pixels actually occur inside the
    mapped area (holes in the study area, or inside the inset window). The white area
    outside the region boundary is not a data class and is no longer labelled.
"""


import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch, Rectangle
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling
from scipy.ndimage import uniform_filter, binary_fill_holes
import matplotlib.patheffects as pe

# ------------------------------------------------------------------ config
MAP_DIR   = DATA_DIR + 'maps/'
MODEL_DIR = RESULTS_DIR + 'train_test_gap/'
OUT_DIR   = MAP_DIR
DISPLAY_RES = 30              # m, full-extent panels
NATIVE_RES  = 10              # m, inset panels
INSET_BOUNDS = None           # (left, bottom, right, top) in map CRS metres; None = auto
INSET_SIZE_KM = 8
MODELS = ['CatBoost', 'XGBoost', 'MLP', 'GTB', 'RF', 'SVM-RBF', 'KNN', 'CART', 'MDC', 'NB']  # same order as all multi-panel figures

# Inferred from class counts (0: 300, 1: 950, 2: 650, 3: 430, 4: 470) - confirm against your GEE labels
CLASS_NAMES  = {0: 'Water', 1: 'Forest', 2: 'Bare ground', 3: 'Urban', 4: 'Crops/grass'}
CLASS_COLORS = {'Water': '#2C7FB8', 'Bare ground': '#C9A27E', 'Crops/grass': '#E6C229',
                'Forest': '#1B7837', 'Urban': '#D7301F'}
NODATA = 255

codes = sorted(CLASS_NAMES)
cmap = ListedColormap([CLASS_COLORS[CLASS_NAMES[c]] for c in codes])
norm = BoundaryNorm(np.arange(min(codes) - 0.5, max(codes) + 1.5), cmap.N)
summary = pd.read_csv(MODEL_DIR + 'train_cv_test_full.csv').set_index('Model')
# Three leading models, CatBoost first (XGBoost, CatBoost and MLP tie at 95.95%; same three as
# the class-error analysis). Change here if the manuscript uses another selection.
TOP3 = ['CatBoost', 'XGBoost', 'GTB']

# One common extent for every model: the 10 m maps and the 30 m maps (KNN, SVM-RBF) cover
# slightly different extents, so without fixed bounds the mosaics differ by a row or column.
def union_bounds(model):
    bs = []
    for f in sorted(glob.glob(MAP_DIR + f'{model}/*.tif')):
        with rasterio.open(f) as src:
            bs.append(src.bounds)
    return (min(b.left for b in bs), min(b.bottom for b in bs),
            max(b.right for b in bs), max(b.top for b in bs))
FULL_BOUNDS = union_bounds(MODELS[0])

def mosaic(model, res, resampling, bounds=None):
    files = sorted(glob.glob(MAP_DIR + f'{model}/*.tif'))
    arr, tr = merge(files, res=res, bounds=bounds or FULL_BOUNDS, nodata=NODATA, resampling=resampling)
    return arr[0], tr

def extent_km(arr, tr):
    h, w = arr.shape
    return [tr.c / 1000, (tr.c + w * tr.a) / 1000, (tr.f + h * tr.e) / 1000, tr.f / 1000]

# ------------------------------------------------------------------ load full-extent maps
disp, samp = {}, {}
for m in MODELS:
    disp[m], tr_disp = mosaic(m, DISPLAY_RES, Resampling.mode)      # for display
    samp[m], _ = mosaic(m, DISPLAY_RES, Resampling.nearest)          # exact 10 m pixels, sampled
    print(f'{m}: {disp[m].shape}')
ext = extent_km(disp[MODELS[0]], tr_disp)

# ------------------------------------------------------------------ agreement
stack = np.stack([samp[m] for m in MODELS])
valid = (stack != NODATA).all(0)
agree = np.max([(stack == c).sum(0) for c in codes], axis=0).astype(float)
agree[~valid] = np.nan
k = agree[valid].astype(int)
agr_tab = pd.Series(np.bincount(k, minlength=len(MODELS) + 1)[1:] / k.size * 100,
                    index=range(1, len(MODELS) + 1), name='% of mapped area').round(2)
agr_tab.index.name = 'models agreeing with majority class'
agr_tab.to_csv(OUT_DIR + 'model_agreement_summary.csv')
print('\n', agr_tab.to_string())
print(f'All {len(MODELS)} agree: {agr_tab.iloc[-1]:.1f}% | '
      f'<= {len(MODELS) - 4} agree: {agr_tab.iloc[:len(MODELS) - 4].sum():.1f}%')

# ------------------------------------------------------------------ inset location
if INSET_BOUNDS is None:
    win = max(3, int(INSET_SIZE_KM * 1000 / DISPLAY_RES))
    dis = np.where(valid, len(MODELS) - np.nan_to_num(agree), 0)
    half = win // 2
    score = (uniform_filter(dis.astype(float), win, mode='constant') *
             (uniform_filter(valid.astype(float), win, mode='constant') > 0.95))
    score[:half + 1, :] = 0; score[-half - 1:, :] = 0     # keep the window inside the map
    score[:, :half + 1] = 0; score[:, -half - 1:] = 0
    r, c = np.unravel_index(np.argmax(score), score.shape)
    x0 = tr_disp.c + (c - win // 2) * tr_disp.a
    y1 = tr_disp.f + (r - win // 2) * tr_disp.e
    INSET_BOUNDS = (x0, y1 - win * DISPLAY_RES, x0 + win * DISPLAY_RES, y1)
    print('\nAuto-selected inset (highest model disagreement):', [round(v) for v in INSET_BOUNDS],
          '\nInspect it; set INSET_BOUNDS by hand if it falls on an artefact or a dull area.')
l, b, rgt, t = INSET_BOUNDS
inset = {m: mosaic(m, NATIVE_RES, Resampling.nearest, bounds=INSET_BOUNDS) for m in MODELS}

# ------------------------------------------------------------------ no-data checks
def nodata_inside(arr):
    """Number of no-data pixels enclosed by mapped pixels (holes inside the study area).
    The empty area outside the region boundary is not counted."""
    v = arr != NODATA
    return int((binary_fill_holes(v) & ~v).sum())

def nodata_in_window(arr):
    """Number of no-data pixels anywhere inside a rectangular window (inset panels)."""
    return int((arr == NODATA).sum())

nd_full  = {m: nodata_inside(disp[m]) for m in MODELS}
nd_inset = {m: nodata_in_window(inset[m][0]) for m in MODELS}
print('\nNo-data pixels inside the study area (full maps):', nd_full)
print('No-data pixels inside the inset window:', nd_inset)

def nodata_needed(models, table):
    return any(table[m] > 0 for m in models)

# ------------------------------------------------------------------ plotting helpers
INSET_EC = 'black'
def draw_inset(ax):
    """Inset square, identical in every panel: black outline with a white halo."""
    rect = Rectangle((l / 1000, b / 1000), (rgt - l) / 1000, (t - b) / 1000,
                     fill=False, ec=INSET_EC, lw=0.9, zorder=5)
    rect.set_path_effects([pe.Stroke(linewidth=2.2, foreground='white'), pe.Normal()])
    ax.add_patch(rect)

def show_map(ax, arr, extent, title, box=True):
    ax.imshow(np.ma.masked_equal(arr, NODATA), cmap=cmap, norm=norm, extent=extent,
              interpolation='nearest')
    if box:
        draw_inset(ax)
    ax.set_title(title, fontsize=7.5, fontweight='bold', pad=3, linespacing=1.1)
    ax.set_xticks([]); ax.set_yticks([])

def scalebar(ax, length_km, extent):
    x0 = extent[0] + 0.05 * (extent[1] - extent[0])
    y0 = extent[2] + 0.05 * (extent[3] - extent[2])
    ax.plot([x0, x0 + length_km], [y0, y0], color='black', lw=2)
    ax.text(x0 + length_km / 2, y0 + 0.02 * (extent[3] - extent[2]), f'{length_km:g} km',
            ha='center', va='bottom', fontsize=6.5,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=1))

INSET_HANDLE = Patch(fc='none', ec=INSET_EC, lw=0.9, label='Inset area')

def legend(fig, extra=None, nodata=False):
    handles = [Patch(fc=CLASS_COLORS[CLASS_NAMES[c]], ec='none', label=CLASS_NAMES[c]) for c in codes]
    if nodata:                       # only when no-data pixels are really visible in the panels
        handles += [Patch(fc='white', ec='#999999', label='No data')]
    handles += (extra or [])
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), frameon=False, fontsize=8)

def title_of(m):
    return f"{m}\nOA {100 * summary.loc[m, 'Test_OA']:.2f}%" if m in summary.index else m

def save(fig, name):
    fig.savefig(OUT_DIR + name + '.png', dpi=600, bbox_inches='tight')
    fig.savefig(OUT_DIR + name + '.pdf', bbox_inches='tight')
    plt.show()

aspect = (ext[3] - ext[2]) / (ext[1] - ext[0])
bar_km = max(1, round((ext[1] - ext[0]) / 5, -1))

# ------------------------------------------------------------------ Fig 1: all ten
pw = 7.1 / 5
fig, axes = plt.subplots(2, 5, figsize=(7.1, 2 * pw * aspect + 1.1))
for ax, m in zip(axes.flat, MODELS):
    show_map(ax, disp[m], ext, title_of(m))
scalebar(axes.flat[0], bar_km, ext)
legend(fig, [INSET_HANDLE], nodata=nodata_needed(MODELS, nd_full))
fig.tight_layout(rect=(0, 0.06, 1, 1), w_pad=0.3, h_pad=1.6)
save(fig, 'Fig_maps_all10')

# ------------------------------------------------------------------ Fig 2: inset, all ten
ins_ext = [l / 1000, rgt / 1000, b / 1000, t / 1000]
ins_aspect = (t - b) / (rgt - l)
fig, axes = plt.subplots(2, 5, figsize=(7.1, 2 * pw * ins_aspect + 0.7))
for ax, m in zip(axes.flat, MODELS):
    show_map(ax, inset[m][0], ins_ext, m, box=False)
scalebar(axes.flat[0], max(1, round((rgt - l) / 1000 / 4)), ins_ext)
legend(fig, nodata=nodata_needed(MODELS, nd_inset))      # no inset square in this figure
fig.tight_layout(rect=(0, 0.06, 1, 1), w_pad=0.3, h_pad=0.6)
save(fig, 'Fig_maps_inset_all10')

# ------------------------------------------------------------------ Fig 3: main-text selection
fig, axes = plt.subplots(2, 2, figsize=(7.1, 2 * 3.55 * aspect + 0.8))
for ax, m in zip(axes.flat[:3], TOP3):
    show_map(ax, disp[m], ext, title_of(m))
ax = axes.flat[3]
acmap = plt.get_cmap('viridis', len(MODELS))
im = ax.imshow(agree, cmap=acmap, vmin=0.5, vmax=len(MODELS) + 0.5, extent=ext,
               interpolation='nearest')
draw_inset(ax)                     # same symbol as in the classification maps and the legend
ax.set_title('Agreement among\nthe ten models', fontsize=7.5, fontweight='bold', pad=3, linespacing=1.1)
ax.set_xticks([]); ax.set_yticks([])
cax = ax.inset_axes([1.03, 0.1, 0.04, 0.8])
cb = fig.colorbar(im, cax=cax, ticks=range(1, len(MODELS) + 1))
cb.set_label('Models agreeing with the majority class', fontsize=7)
cb.ax.tick_params(labelsize=6.5)
scalebar(axes.flat[0], bar_km, ext)
legend(fig, [INSET_HANDLE], nodata=nodata_needed(TOP3, nd_full))
fig.tight_layout(rect=(0, 0.05, 0.92, 1))   # leave room for the colour bar
save(fig, 'Fig_maps_main')
print('\nFiles saved to:', OUT_DIR)

In [ ]:
# Colab-only helper: downloads the figures to your computer. Skip this cell when running locally.
import os, time
from google.colab import files, drive

OUT_DIR = DATA_DIR + 'maps/'
names = ['Fig_maps_main', 'Fig_maps_all10', 'Fig_maps_inset_all10']

# 1. Check the files on disk: size and time they were last written
for n in names:
    p = OUT_DIR + n + '.png'
    print(n, '|', round(os.path.getsize(p) / 1e6, 1), 'MB |',
          time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))))

# 2. Download the PNGs directly to your computer (bypasses Drive)
for n in names:
    files.download(OUT_DIR + n + '.png')

# 3. Force Colab to finish writing everything to Google Drive
drive.flush_and_unmount()
print('Drive synced.')

In [ ]:
# Colab-only helper: downloads the figures to your computer. Skip this cell when running locally.
from google.colab import drive, files
import os, shutil

drive.mount('/content/drive')

SRC = DATA_DIR + 'maps/'
DST = RESULTS_DIR + 'Figures_revised_14Oct/'
os.makedirs(DST, exist_ok=True)

for n in ['Fig_maps_main', 'Fig_maps_all10', 'Fig_maps_inset_all10']:
    shutil.copy2(SRC + n + '.png', DST + n + '_v2.png')
    shutil.copy2(SRC + n + '.pdf', DST + n + '_v2.pdf')
print(os.listdir(DST))

# One zip, built on the Colab machine itself, then downloaded
shutil.make_archive(RESULTS_DIR + 'Figures_revised', 'zip', DST)
files.download(RESULTS_DIR + 'Figures_revised.zip')